# Training Log Reader
Parst `traininglog_test.docx` in einen pandas DataFrame und berechnet die Trainingsbelastung.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.traininglog_parser import parse_training_log
from src.load_score import compute_endurance_performance, compute_combined_load

In [2]:
FILE_PATH = r"e:\GoogleDrive\privat\Code_Repo\AI_Coach\data\traininglog_test.docx"

In [8]:
# Inspect raw docx structure (diagnostic)
from docx import Document

doc = Document(FILE_PATH)

print("=== PARAGRAPHS ===")
for i, para in enumerate(doc.paragraphs):
    if para.text.strip():
        print(f"[{i}] Style='{para.style.name}' | {para.text}")

print("\n=== TABLES ===")
for t_idx, table in enumerate(doc.tables):
    print(f"\n-- Table {t_idx} ({len(table.rows)} rows x {len(table.columns)} cols) --")
    for row in table.rows:
        print([cell.text.strip() for cell in row.cells])

=== PARAGRAPHS ===
[0] Style='Normal' | Date: 20.04.2026
[1] Style='List Paragraph' | Activity 1:
[2] Style='List Paragraph' | Bike:
[3] Style='List Paragraph' | Steady-State:
[4] Style='List Paragraph' | Duration [min]: 25
[5] Style='List Paragraph' | Hear-Rate [bpm]: 115-145
[6] Style='List Paragraph' | Speed [km/h]: 
[7] Style='List Paragraph' | Activity 2:
[8] Style='List Paragraph' | Bike:
[9] Style='List Paragraph' | Sprints: 
[10] Style='List Paragraph' | Sets: 3
[11] Style='List Paragraph' | Duration [s]: 6
[12] Style='List Paragraph' | Hear-Rate [bpm]:
[13] Style='List Paragraph' | Speed [km/h]:
[14] Style='List Paragraph' | Rest between sets [s]: 90
[15] Style='Normal' | Date: 22.04.2026
[16] Style='List Paragraph' | Activity 1:
[17] Style='List Paragraph' | Running:
[18] Style='List Paragraph' | Intervalls: 
[19] Style='List Paragraph' | Sets: 3
[20] Style='List Paragraph' | Duration [s]: 120
[21] Style='List Paragraph' | Hear-Rate [bpm]: 115-145
[22] Style='List Paragraph' 

In [9]:
df = parse_training_log(FILE_PATH)
display(df)

,date,activity_nr,sport,training_type,duration_[min],hear_rate_[bpm],speed_[km/h],sets,duration_[s],rest_between_sets_[s],pace_[min/km],repetitions
0,2026-04-20,Activity 1,Bike,Steady-State,25,115-145,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-20,Activity 2,Bike,Sprints,NaN,NaN,NaN,3,6,90,NaN,NaN
2,2026-04-22,Activity 1,Running,Intervalls,NaN,115-145,NaN,3,120,60,NaN,NaN
3,2026-04-22,Activity 2,Speed Drills,Skippings,NaN,NaN,NaN,3,6,90,NaN,NaN
4,2026-04-22,Activity 3,Sprint,Intervalls,NaN,NaN,NaN,3,6,90,NaN,NaN
5,2026-04-22,Activity 4,Running,Intervalls,NaN,150-190,NaN,2,40,180,NaN,NaN
6,2026-04-25,Activity 1,Bike,Steady-State,30,115-145,NaN,NaN,NaN,NaN,NaN,NaN
7,2026-04-25,Activity 2,Speed Drills,Skippings,NaN,NaN,NaN,3,6,90,NaN,NaN
8,2026-04-25,Activity 3,Power,Jumps,NaN,NaN,NaN,3,NaN,90,NaN,3
9,2026-04-25,Activity 4,Bike,Steady-State,20,115-145,NaN,NaN,NaN,NaN,NaN,NaN


## Trainingsbelastung

### Kardiovaskuläre Belastung: TRIMP (Banister)

$$\text{TRIMP} = t \cdot \Delta\text{HR} \cdot e^{b \cdot \Delta\text{HR}}, \quad \Delta\text{HR} = \frac{\overline{\text{HR}} - \text{HR}_\text{rest}}{\text{HR}_\text{max} - \text{HR}_\text{rest}}$$

- **$t$**: aktive Trainingszeit [min] — $b$: Geschlechtsfaktor (1,92 = männlich / 1,67 = weiblich)
- Ohne HR-Daten: TRIMP = Dauer [min] als Volumen-Proxy

### Gesamtbelastung (sport-korrigierter TRIMP)

$$\text{Gesamtbelastung} = \text{TRIMP} \times k_\text{Sport}$$

| Sport | $k_\text{Sport}$ | Grund |
|---|---|---|
| Radfahren (Referenz) | 1,0 | – |
| Laufen | 1,3 | Bodenreaktionskräfte, exzentrische Muskelbelastung |
| Schwimmen | 1,1 | Schultergürtel-Overhead |

> **Nächster Schritt:** `compute_combined_load(df_session, df_wellness)` in `src/load_score.py`
> kombiniert TRIMP mit Garmin-Erholungsmetriken (Resting HR, HRV) zu einem Readiness-Score.

In [10]:
df_session, df_weekly = compute_endurance_performance(df)

print("=== Belastung pro Trainingseinheit ===")
display(df_session)

print("\n=== Wochensummen ===")
display(df_weekly)

=== Belastung pro Trainingseinheit ===


,date,year,calendar_week,activity_nr,sport,training_type,total_duration_min,avg_hr_bpm,trimp,sport_factor,total_load
0,2026-04-20,2026,17,Activity 1,Bike,Steady-State,25.000000,130.0,37.851960,1.0,37.851960
1,2026-04-20,2026,17,Activity 2,Bike,Sprints,0.300000,NaN,0.300000,1.0,0.300000
2,2026-04-22,2026,17,Activity 1,Running,Intervalls,6.000000,130.0,9.084470,1.0,9.084470
3,2026-04-22,2026,17,Activity 2,Speed Drills,Skippings,0.300000,NaN,0.300000,1.0,0.300000
4,2026-04-22,2026,17,Activity 3,Sprint,Intervalls,0.300000,NaN,0.300000,1.0,0.300000
5,2026-04-22,2026,17,Activity 4,Running,Intervalls,1.333333,170.0,5.727295,1.0,5.727295
6,2026-04-25,2026,17,Activity 1,Bike,Steady-State,30.000000,130.0,45.422352,1.0,45.422352
7,2026-04-25,2026,17,Activity 2,Speed Drills,Skippings,0.300000,NaN,0.300000,1.0,0.300000
8,2026-04-25,2026,17,Activity 3,Power,Jumps,NaN,NaN,NaN,1.0,NaN
9,2026-04-25,2026,17,Activity 4,Bike,Steady-State,20.000000,130.0,30.281568,1.0,30.281568



=== Wochensummen ===


,year,calendar_week,sessions,total_duration_min,trimp_weekly,total_load_weekly
0,2026,17,9,83.533333,129.567647,129.567647
1,2026,18,13,149.400000,237.979824,237.979824


In [3]:

import sys
sys.path.insert(0, "..")
from src.traininglog_parser import parse_training_log
from src.load_score import compute_endurance_performance
import re, numpy as np

df = parse_training_log(FILE_PATH)
df_session, _ = compute_endurance_performance(df)

# Zeige alle Felder für Intervall-Sessions (sets vorhanden)
hr_col = next((c for c in ("hear_rate_[bpm]", "heart_rate_[bpm]") if c in df.columns), None)
interval_rows = df[df.get("sets", df.get("sets", None)).notna()] if "sets" in df.columns else df.iloc[0:0]

cols_show = ["date", "sport", "training_type", hr_col, "sets", "duration_[s]", "duration_[min]"]
cols_show = [c for c in cols_show if c and c in df.columns]
print("=== Rohdaten Intervall-Sessions ===")
print(interval_rows[cols_show].to_string())

print("\n=== Berechnete Werte Intervall-Sessions ===")
idx = interval_rows.index
print(df_session.loc[idx][["date","sport","training_type","total_duration_min","avg_hr_bpm","trimp","sport_factor","total_load"]].to_string())


=== Rohdaten Intervall-Sessions ===
         date         sport training_type hear_rate_[bpm] sets duration_[s] duration_[min]
1  2026-04-20          Bike       Sprints             NaN    3            6            NaN
2  2026-04-22       Running    Intervalls         115-145    3          120            NaN
3  2026-04-22  Speed Drills     Skippings             NaN    3            6            NaN
4  2026-04-22        Sprint    Intervalls             NaN    3            6            NaN
5  2026-04-22       Running    Intervalls         150-190    2           40            NaN
7  2026-04-25  Speed Drills     Skippings             NaN    3            6            NaN
8  2026-04-25         Power         Jumps             NaN    3          NaN            NaN
11 2026-04-28  Speed Drills     Skippings             NaN    3            6            NaN
12 2026-04-28         Power         Jumps             NaN    3          NaN            NaN
13 2026-04-28          Bike       Sprints             

In [5]:

import sys; sys.path.insert(0, "..")
from src.traininglog_parser import parse_training_log

df_raw = parse_training_log(FILE_PATH)
hr_col = next((c for c in ("hear_rate_[bpm]", "heart_rate_[bpm]") if c in df_raw.columns), None)

if "sets" in df_raw.columns:
    mask = df_raw["sets"].notna()
    sub = df_raw[mask][["date","sport","training_type", hr_col, "sets", "duration_[s]"]].copy()
    print(f"HR-Spalte: {hr_col!r}")
    print(sub.to_string(index=False))
else:
    print("Keine 'sets'-Spalte – keine Intervall-Sessions erkannt")
    print(df_raw.columns.tolist())


HR-Spalte: 'hear_rate_[bpm]'
      date        sport training_type hear_rate_[bpm] sets duration_[s]
2026-04-20         Bike       Sprints             NaN    3            6
2026-04-22      Running    Intervalls         115-145    3          120
2026-04-22 Speed Drills     Skippings             NaN    3            6
2026-04-22       Sprint    Intervalls             NaN    3            6
2026-04-22      Running    Intervalls         150-190    2           40
2026-04-25 Speed Drills     Skippings             NaN    3            6
2026-04-25        Power         Jumps             NaN    3          NaN
2026-04-28 Speed Drills     Skippings             NaN    3            6
2026-04-28        Power         Jumps             NaN    3          NaN
2026-04-28         Bike       Sprints             NaN    3            6
2026-04-30 Speed Drills     Skippings             NaN    3            6
2026-04-30       Sprint    Intervalls             NaN    4            6
2026-05-02 Speed Drills     Skippin